# 02 · ALS Recommendation Model — Amazon Movies & TV

Builds a collaborative filtering model using PySpark MLlib ALS.  
Input: user star ratings · Output: personalised movie/show recommendations.

In [ ]:
# ── Update to your Databricks volume path ─────────────────────────────────────
DATA_DIR = "/Volumes/workspace/ds_amazonrec/data"
# ─────────────────────────────────────────────────────────────────────────────

## 1 · Load & Filter Data

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.ml.recommendation import ALS

raw_reviews = spark.read.json(f"{DATA_DIR}/reviews_part*.jsonl")

meta_schema = StructType([
    StructField("parent_asin", StringType(), True),
    StructField("title",       StringType(), True),
])
raw_meta = spark.read.schema(meta_schema).json(f"{DATA_DIR}/meta_Movies_and_TV.jsonl")

reviews = raw_reviews.select(
    F.col("user_id"),
    F.col("parent_asin").alias("product_id"),
    F.col("rating").cast("float"),
    F.col("timestamp"),
)

products = raw_meta.select(
    F.col("parent_asin").alias("product_id"),
    F.col("title").alias("name"),
).dropDuplicates(["product_id"])

print(f"Raw: {reviews.count():,} reviews")

In [ ]:
# 5-core filter: keep only users and products with >= 5 interactions
# max_iter=4 is enough for this dataset; count checks removed for Serverless performance
def apply_k_core(df, k=5, max_iter=4):
    for i in range(max_iter):
        product_counts = df.groupBy("product_id").count()
        df = df.join(product_counts.filter(F.col("count") >= k).select("product_id"), "product_id")
        user_counts = df.groupBy("user_id").count()
        df = df.join(user_counts.filter(F.col("count") >= k).select("user_id"), "user_id")
        print(f"Iter {i+1} done")
    return df

reviews_filtered = apply_k_core(reviews)
print("5-core filtering complete")

## 2 · Index String IDs → Integers

Use `dense_rank()` instead of `StringIndexer` to avoid the 256 MB model size limit on Databricks Serverless.

In [ ]:
from pyspark.sql import Window as W

# Assign 0-based integer indices via dense_rank (no model fitting, no size limits)
user_window    = W.orderBy("user_id")
product_window = W.orderBy("product_id")

indexed = reviews_filtered \
    .withColumn("userIdx",    (F.dense_rank().over(user_window)    - 1).cast("integer")) \
    .withColumn("productIdx", (F.dense_rank().over(product_window) - 1).cast("integer"))

# Build productIdx -> product_id lookup for display later
idx_to_product = {
    row["productIdx"]: row["product_id"]
    for row in indexed.select("productIdx", "product_id").distinct().collect()
}
products_pd = products.toPandas().set_index("product_id")["name"].to_dict()

n_users    = indexed.select("userIdx").distinct().count()
n_products = indexed.select("productIdx").distinct().count()
print(f"Users   : {n_users:,}")
print(f"Products: {n_products:,}")
indexed.select("user_id", "userIdx", "product_id", "productIdx", "rating").show(5)

## 3 · Train / Test Split

In [ ]:
train, test = indexed.randomSplit([0.8, 0.2], seed=42)
# train.cache(); test.cache()  # disabled: cache() causes issues on Databricks Serverless
print(f"Train: {train.count():,} | Test: {test.count():,}")

## 4 · Train ALS Model

In [ ]:
import time

als = ALS(
    rank=50,
    maxIter=10,
    regParam=0.1,
    userCol="userIdx",
    itemCol="productIdx",
    ratingCol="rating",
    implicitPrefs=False,
    coldStartStrategy="drop",
    seed=42
)

t0 = time.time()
als_model = als.fit(train)
print(f"ALS trained in {time.time()-t0:.1f}s  (rank={als.getRank()})")

## 5 · Find Similar Movies

Cosine similarity on ALS item factors.  
Note: ALS learns *user behaviour overlap*, not content similarity —
results reflect which users co-rated items, not what the items are about.

In [ ]:
import math
import pandas as pd

item_factors_pd = als_model.itemFactors.toPandas().rename(columns={"id": "productIdx"})
item_factors_pd["product_id"] = item_factors_pd["productIdx"].map(idx_to_product)
item_factors_pd["name"]       = item_factors_pd["product_id"].map(products_pd)

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na  = math.sqrt(sum(x*x for x in a))
    nb  = math.sqrt(sum(x*x for x in b))
    return dot / (na * nb) if na * nb > 0 else 0.0

def similar_movies(query_name, top_n=8):
    mask = item_factors_pd["name"].str.lower().str.contains(query_name.lower(), na=False)
    if not mask.any():
        print(f'"{query_name}" not found')
        return
    row  = item_factors_pd[mask].iloc[0]
    sims = [cosine(row["features"], f) for f in item_factors_pd["features"]]
    item_factors_pd["sim"] = sims
    top  = item_factors_pd[item_factors_pd["name"] != row["name"]].nlargest(top_n, "sim")[["name", "sim"]]
    print(f'\nMovies/shows similar to "{row["name"]}":')
    print(f'{"Title":<50} {"Similarity":>10}')
    print("-" * 62)
    for _, r in top.iterrows():
        print(f'{str(r["name"])[:50]:<50} {r["sim"]:>10.4f}')

In [ ]:
similar_movies("inception")
similar_movies("breaking bad")
similar_movies("harry potter")
similar_movies("two and a half men")

## 6 · Recommend for a Specific User

Selects the most active user in the training set (most ratings),
then generates top-10 recommendations for unseen items.

In [ ]:
from pyspark.sql import Window

# Pick the most active user (highest review count) for a richer demonstration
top_user        = train.groupBy("userIdx", "user_id").count().orderBy(F.col("count").desc()).first()
sample_user_idx = top_user["userIdx"]   # integer index used by ALS
sample_user_id  = top_user["user_id"]   # original string ID for display

already_rated = (
    train.filter(F.col("userIdx") == sample_user_idx)
    .join(products, "product_id", "left")
    .select("name", "rating")
    .orderBy(F.col("rating").desc())
)
print(f"User {sample_user_id} — already rated:")
already_rated.show(10, truncate=50)

user_df    = spark.createDataFrame([(sample_user_idx,)], ["userIdx"])
all_items  = indexed.select("productIdx").distinct()
seen_items = train.filter(F.col("userIdx") == sample_user_idx).select("productIdx")
unseen     = all_items.subtract(seen_items)
window     = Window.partitionBy("userIdx").orderBy(F.col("prediction").desc())

# Score all unseen items, take top 50 by prediction, then filter to those with metadata
recs = (
    als_model.transform(user_df.crossJoin(unseen))
    .filter(F.col("prediction").isNotNull())
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") <= 50)
)

# Join back to product metadata — some productIdx have no metadata, so filter nulls
recs_named = (
    recs
    .join(indexed.select("productIdx", "product_id").distinct(), "productIdx", "left")
    .join(products, "product_id", "left")
    .filter(F.col("name").isNotNull())
    .orderBy("rank")
    .limit(10)
    .select("rank", "name", F.round("prediction", 3).alias("predicted_rating"))
)

print(f"\nTop-10 recommendations for user {sample_user_id}:")
recs_named.show(10, truncate=60)